### Import libraries.

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
import pandas as pd
import os
import requests
import json
import csv
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Implemented file handling functions.

In [ ]:
def download_image(source, filename):
    with open(filename, 'wb') as f:
        image_content = requests.get(source).content
        f.write(image_content)
        print('Downloaded image at:', filename)

def write_txt(filename, data):
    with open(filename, 'w', encoding='utf-8') as f:
        f.writelines(data)
    return True


def read_txt(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = f.readlines()
    return data


def write_json(filename, data):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    return True


def read_json(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


def write_csv(filename, data):
    with open(filename, 'w', encoding='utf-8', newline='') as f:
        csv_writer = csv.writer(f)
        for row in data:
            csv_writer.writerow(row)
    return True


def read_csv(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        title = next(reader)
        data = [row for row in reader]
    return title, data

### Extracted course URLs from web pages.

In [ ]:
driver = webdriver.Chrome()
driver.maximize_window()
driver.get("https://www.coursera.org/search?query=data%20science&language=English&topic=Data%20Science&sortBy=BEST_MATCH")

time.sleep(3)

# Get the initial page height
last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    # Scroll down by 500 pixels
    driver.execute_script('window.scrollBy(0, 500)')
    time.sleep(1)  # Wait for new content to load

    # Get the current vertical scroll position
    new_height = driver.execute_script("return window.scrollY")

    #  If the scroll position does not change, no new content is loaded -> stop the loop
    if new_height == last_height:
        break
    
    # Update the last scroll position for the next iteration
    last_height = new_height

##### Description of the auto-scrolling mechanism for loading and extracting course links:

**1.** Initialize the page height: 'last_height = driver.execute_script("return document.body.scrollHeight")'

**2.** Scroll down to dynamically load more content: 
* Infinite loop (while True): Continuously scrolls the page until no new content is loaded.
* Scroll action: driver.execute_script('window.scrollBy(0, 500)'): Executes JavaScript to scroll down by 500 pixels.Wait for content to load:
* time.sleep(1): Pauses for 1 second to allow new content to load. 
* new_height = driver.execute_script("return window.scrollY"): Get current scroll position. 
* Stopping condition:: If the current scroll position remains unchanged compared to the previous iteration, it indicates that no additional content is being loaded → stop the loop. 
* Update last position: Update last_height for comparison in the next iteration. 
**==>** With a scrolling step of 500 pixels per second, after approximately 6 minutes, the page successfully loads around 1000 course results.


In [84]:
page_source = driver.page_source
page_content = BeautifulSoup(page_source, 'html.parser')
a = page_content.find_all(name='a', attrs={'data-click-key':'search.search.click.search_card'})

**1,** Use driver.page_source to retrieve the fully loaded HTML content of the page.

**2*.* Use the BeautifulSoup (bs4) library to extract all product links from <a> tags with the attribute data-click-key="search.search.click.search_card".

**>>** The extracted links only contain relative paths (file paths) and do not include the base domain.

In [ ]:
list_ = []
for i in a:
    j = 'https://www.coursera.org' + i['href'] + '\n'
    list_.append(j)
write_txt('list_coursera_links.txt', list_)
print('Save successfully!!!')

Lưu thành công!!!


**1.** Initialize an empty list:
* list_ = [] creates an empty Python list named list_ to store the constructed links.

**2.** Process the extracted links:
* for i in a: iterates through each element i in the list a, which contains all <a> tags extracted from the webpage.
* Concatenate the base URL 'https://www.coursera.org' (the root domain)
* Append the href attribute of each element i
* Add a newline character (\n) to separate links when saving to a text file
* list_.append(j) adds the constructed string j to the end of the list_

**3.** Save the list to a file:

### Scrape data from course URLs.

In [ ]:
driver = webdriver.Chrome()
driver.maximize_window()

list_links = read_txt('list_coursera_links.txt')

for i in range(0, len(list_links)):
    link = list_links[i]
    page_link = link.rstrip('\n')
    
    driver.get(url=page_link)
    time.sleep(2)

    # Organization name
    try:
        org_name = driver.find_element(by=By.CSS_SELECTOR, value='img.css-1f9gt0j').get_attribute('alt')
    except:
        org_name = 'Nan'

    # Course name
    try:
        coursera_name = driver.find_element(by=By.CSS_SELECTOR, value='h1.cds-119.cds-Typography-base.css-1xy8ceb.cds-121').text
    except:
        coursera_name = 'Nan'

    # Course code
    try:
        coursera_code = driver.find_element(by=By.CSS_SELECTOR, value='h1.cds-119.cds-Typography-base.css-1xy8ceb.cds-121').text
    except:
        coursera_code = 'Nan'

    # Average rating (stars)
    try:
        stars_html = driver.find_element(by=By.CSS_SELECTOR, value='div.cds-119.cds-Typography-base.css-h1jogs.cds-121').get_attribute('outerHTML')
        soup = BeautifulSoup(stars_html, 'html.parser')
        stars = soup.get_text()
    except:
        stars = 'NaN'

    # Number of reviews
    try:
        elements_reviews = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//p[contains(text(), "reviews")]'))
        )
        # Extract text content from elements containing "reviews"
        for element in elements_reviews:
            reviews = element.text.replace('(', '').replace(')', '').replace(',', '').split(' ')[0].rstrip('')
    except:
        reviews = 'NaN'

    # Course level
    try:
        elements_level = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//div[contains(text(), "level")]'))
        )
        # Find element with the specific class and extract level information
        for element in elements_level:
            if ' css-fk6qfz' in element.get_attribute('class'):
                level = element.text
    except:
        level = 'Nan'

    # Learning outcomes
    try:
        gain = driver.find_elements(
            by=By.XPATH,
            value='//*[@id="about"]/div/div[1]/div[2]/div/ul/li/div/div/div/p/span/span'
        )
        gain_content = []
        for content in gain:
            html_content = content.text
            gain_content.append(html_content)
    except:
        gain_content = []

    
    # Create course dictionary
    dict_coursera = {
        'organization_name': org_name,
        'coursera_name': coursera_name,
        'coursera_code': coursera_code,
        'star': stars,
        'reviews': reviews,
        'level': level,
        'result': gain_content
    }

    json_path = f'data/coursera/{i + 1}.json'
    write_json(json_path, dict_coursera)

    print(f'=>>> Saved {json_path}')

else:
    print('>>> Completed!!!')
    driver.close()

=>>> Đã lưu data/coursera/865.json
=>>> Đã lưu data/coursera/866.json
=>>> Đã lưu data/coursera/867.json
=>>> Đã lưu data/coursera/868.json
=>>> Đã lưu data/coursera/869.json
=>>> Đã lưu data/coursera/870.json
=>>> Đã lưu data/coursera/871.json
=>>> Đã lưu data/coursera/872.json
=>>> Đã lưu data/coursera/873.json
=>>> Đã lưu data/coursera/874.json
=>>> Đã lưu data/coursera/875.json
=>>> Đã lưu data/coursera/876.json
=>>> Đã lưu data/coursera/877.json
=>>> Đã lưu data/coursera/878.json
=>>> Đã lưu data/coursera/879.json
=>>> Đã lưu data/coursera/880.json
=>>> Đã lưu data/coursera/881.json
=>>> Đã lưu data/coursera/882.json
=>>> Đã lưu data/coursera/883.json
=>>> Đã lưu data/coursera/884.json
=>>> Đã lưu data/coursera/885.json
=>>> Đã lưu data/coursera/886.json
=>>> Đã lưu data/coursera/887.json
=>>> Đã lưu data/coursera/888.json
=>>> Đã lưu data/coursera/889.json
=>>> Đã lưu data/coursera/890.json
=>>> Đã lưu data/coursera/891.json
=>>> Đã lưu data/coursera/892.json
=>>> Đã lưu data/cou

#### Description of how the code works to extract course information from the list of previously collected links:
##### Since some links do not have the same HTML structure as most others, the code above uses a try-except approach. The goal is to minimize time spent debugging each page and optimize the data scraping process. 
#### Most pages share a similar structure in terms of the tags containing information. Therefore, we will skip pages with special structures (e.g., page number 21) and only extract available information. Missing values will be set to 'Nan' and handled later during the DataFrame processing stage.

**Organization Name** : Use find_element with By.CSS_SELECTOR to extract content from the tag 'img.css-1f9gt0j' using the alt attribute.

**Coursera Name** : Use find_element with By.CSS_SELECTOR to extract content from the tag h1.cds-119.cds-Typography-base.css-1xy8ceb.cds-121.

**Coursera Code** : Use find_element with By.CSS_SELECTOR to extract content from the tag h1.cds-119.cds-Typography-base.css-1xy8ceb.cds-121.

**Star** : Using find_element with By.CSS_SELECTOR on the tag div.cds-119.cds-Typography-base.css-h1jogs.cds-121 inside the loop may fail. → Use BeautifulSoup (bs4) to extract the HTML content and retrieve the value instead.

**Reviews** : The tag containing this value is defined differently compared to others, e.g., p.class=" css-vac8rf", making it difficult to access directly. Therefore, we search for all tags containing the text 'reviews'. From there, we identify the relevant elements and extract the desired content from the p tag.

**Level** : Similar to Reviews, the level is defined as div.class=" css-fk6qfz", and there are many such tags on the page containing irrelevant information. Therefore, we search for elements containing the text 'level', then filter and extract the correct div tag with class=" css-fk6qfz".

**Result** : The structure includes li tags nested through 3 div tags, 1 p tag, and 2 span tags before reaching the desired content → Use a loop with the XPath:
'//*[@id="about"]/div/div[1]/div[2]/div/ul/li/div/div/div/p/span/span'
to directly access each li element and extract all required content.

##### ->> Then, create a dictionary dict_coursera to store the extracted values and save them into numbered JSON files.

### Merge JSON files and convert them into a CSV file.

* Use the learned functions to save the result as coursera_full.csv.

In [ ]:

list_organization_names = []
list_coursera_names = []
list_coursera_codes = []
list_stars = []
list_reviews = []
list_levels = []
list_result = []

coursera_json_folder = 'data/coursera'
for file in os.listdir(coursera_json_folder):
    json_path = f'{coursera_json_folder}/{file}'
    dict_coursera = read_json(json_path)

    list_organization_names.append(dict_coursera['organization_name'])
    list_coursera_names.append(dict_coursera['coursera_name'])
    list_coursera_codes.append(dict_coursera['coursera_code'])
    list_stars.append(dict_coursera['star'])
    list_reviews.append(dict_coursera['reviews'])
    list_levels.append(dict_coursera['level'])
    list_result.append(dict_coursera['result'])
else:
    df_couseras = pd.DataFrame({
        'organization_name' : list_organization_names,
        'coursera_name' : list_coursera_names,
        'coursera_code' : list_coursera_codes,
        'star' : list_stars,

        'reviews' : list_reviews,
        'level' : list_levels,
        'result' : list_result   
    })
    df_couseras.to_csv('coursera_full.csv', index=False)
    print('Completed !!!')

Đã hoàn thành !!!


### Merge JSON files into a single JSON file.

In [ ]:

coursera_json_folder = 'data/coursera'

list_courseras = []

for file in os.listdir(coursera_json_folder):
    json_path = f'{coursera_json_folder}/{file}'
    dict_coursera = read_json(json_path) 
    list_courseras.append(dict_coursera)
else:
    write_json('coursera_full.json', list_courseras)
    print('Completed !!!')

Đã lưu thành công!!!


### Since the results in coursera_full are not ordered according to the JSON file sequence ->> sort them in the correct order.

In [ ]:
list_organization_names = []
list_coursera_names = []
list_coursera_codes = []
list_stars = []
list_reviews = []
list_levels = []
list_result = []

coursera_json_folder = 'data/coursera'
files = os.listdir(coursera_json_folder)

# Sắp xếp theo số thứ tự đã đánh dấu
files.sort(key=lambda x: int(os.path.splitext(x)[0]))

for file in files:
    json_path = f'{coursera_json_folder}/{file}'
    dict_coursera = read_json(json_path)

    list_organization_names.append(dict_coursera['organization_name'])
    list_coursera_names.append(dict_coursera['coursera_name'])
    list_coursera_codes.append(dict_coursera['coursera_code'])
    list_stars.append(dict_coursera['star'])
    list_reviews.append(dict_coursera['reviews'])
    list_levels.append(dict_coursera['level'])
    list_result.append(dict_coursera['result'])
else:
    df_courseras = pd.DataFrame({
        'organization_name': list_organization_names,
        'coursera_name': list_coursera_names,
        'coursera_code': list_coursera_codes,
        'star': list_stars,
        'reviews': list_reviews,
        'level': list_levels,
        'result': list_result   
    })
    df_courseras.to_csv('coursera_full_sorted.csv', index=False)
    print('Completed !!!')

Đã hoàn thành !!!


### Similarly, coursera_full.json is not ordered yet ->> sort it in the correct order.

In [ ]:
coursera_json_folder = 'data/coursera'
list_courseras = []

files = os.listdir(coursera_json_folder)
files.sort(key=lambda x: int(os.path.splitext(x)[0]))

for file in files:
    json_path = f'{coursera_json_folder}/{file}'
    dict_coursera = read_json(json_path)
    list_courseras.append(dict_coursera)

write_json('coursera_full_sorted.json', list_courseras)
print('Completed !!!')

Đã lưu thành công!!!
